# ERA5 Standardized Precipitation Index

This Colab-oriented reference downloads ERA5 monthly averaged total precipitation, converts the mean daily accumulation to monthly accumulated precipitation, fits the historical distribution, and calculates SPI. Configure CDS credentials before running the download.

In [ ]:
!pip -q install cdsapi xarray xclim dask cftime netCDF4

## Imports and configuration

In [ ]:
import logging
from contextlib import ExitStack
from datetime import date, datetime, timezone
from pathlib import Path

import cdsapi
import xarray as xr
from xclim.indices import standardized_precipitation_index
from xclim.indices.stats import standardized_index_fit_params

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("era5-spi")

In [ ]:
BASE_PATH = Path("/content/riskclima-spi")
DOWNLOAD_START = date(1940, 1, 1)
DOWNLOAD_END = date(2026, 7, 1)
RAW_FILE = BASE_PATH / f"era5_tp_monthly_{DOWNLOAD_START.isoformat()}_{DOWNLOAD_END.isoformat()}.nc"
OUTPUT_FILE = BASE_PATH / f"spi1_era5_{DOWNLOAD_START.isoformat()}_{DOWNLOAD_END.isoformat()}.nc"

DATASET = "reanalysis-era5-single-levels-monthly-means"
PRODUCT_TYPE = "monthly_averaged_reanalysis"
REQUEST_VARIABLE = "total_precipitation"
AREA = [20, -120, -70, -5]
CALIBRATION_START = "1961-01-01"
CALIBRATION_END = "1990-12-31"
APPLICATION_START = DOWNLOAD_START.isoformat()
APPLICATION_END = DOWNLOAD_END.isoformat()
SPI_SCALE_MONTHS = 1

## Download the complete and current-year portions

Every run downloads temporary `.part.nc` files, validates and concatenates them, and atomically replaces the exact configured final file. Temporary files are removed after success or failure. Distinct configured periods naturally use distinct date-based filenames.

In [ ]:
def request(years, months):
    return {
        "product_type": [PRODUCT_TYPE],
        "variable": [REQUEST_VARIABLE],
        "year": [str(year) for year in years],
        "month": [f"{month:02d}" for month in months],
        "time": ["00:00"],
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": AREA,
    }


def standardize_era5_dims(dataset):
    rename = {
        source: target
        for source, target in {"valid_time": "time", "latitude": "lat", "longitude": "lon"}.items()
        if source in dataset.dims or source in dataset.coords
    }
    return dataset.rename(rename).sortby(["time", "lat", "lon"])

In [ ]:
complete_end_year = DOWNLOAD_END.year if DOWNLOAD_END.month == 12 else DOWNLOAD_END.year - 1
requests = []
if DOWNLOAD_START.year <= complete_end_year:
    requests.append(request(range(DOWNLOAD_START.year, complete_end_year + 1), range(1, 13)))
if DOWNLOAD_END.month < 12:
    requests.append(request([DOWNLOAD_END.year], range(1, DOWNLOAD_END.month + 1)))

expected_months = [
    (year, month)
    for year in range(DOWNLOAD_START.year, DOWNLOAD_END.year + 1)
    for month in range(1, 13)
    if (year, month) <= (DOWNLOAD_END.year, DOWNLOAD_END.month)
]
RAW_FILE.parent.mkdir(parents=True, exist_ok=True)
labels = ["complete-years", "current-year"] if len(requests) == 2 else ["complete-period"]
part_files = [RAW_FILE.with_suffix(f".{label}.part.nc") for label in labels]
temporary_file = RAW_FILE.with_suffix(f"{RAW_FILE.suffix}.tmp")
client = cdsapi.Client()
try:
    for current_request, part_file in zip(requests, part_files, strict=True):
        client.retrieve(DATASET, current_request, str(part_file))
    with ExitStack() as stack:
        parts = [standardize_era5_dims(stack.enter_context(xr.open_dataset(path))) for path in part_files]
        reference = parts[0]
        if any(not reference.lat.equals(part.lat) or not reference.lon.equals(part.lon) for part in parts[1:]):
            raise ValueError("ERA5 request parts must have exactly equal latitude and longitude grids")
        combined = xr.concat(parts, dim="time", join="exact").sortby(["time", "lat", "lon"])
        months = list(zip(combined.time.dt.year.values, combined.time.dt.month.values))
        if months != expected_months:
            raise ValueError("ERA5 download does not contain every configured month")
        combined.attrs.update(era5_dataset=DATASET, era5_product_type=PRODUCT_TYPE, era5_request_variable=REQUEST_VARIABLE, era5_download_start=DOWNLOAD_START.isoformat(), era5_download_end=DOWNLOAD_END.isoformat(), era5_area=str(AREA))
        combined.to_netcdf(temporary_file)
    temporary_file.replace(RAW_FILE)
finally:
    temporary_file.unlink(missing_ok=True)
    for part_file in part_files:
        part_file.unlink(missing_ok=True)
logger.info("ERA5 input file created successfully: %s", RAW_FILE)

## Convert to monthly accumulation and calculate SPI

In [ ]:
dataset = xr.open_dataset(RAW_FILE)
dataset = standardize_era5_dims(dataset)
tp = dataset["tp"]
monthly_precipitation = (tp * 1000 * tp.time.dt.days_in_month).rename("pr")
monthly_precipitation.attrs["units"] = "mm month-1"

calibration = monthly_precipitation.sel(time=slice(CALIBRATION_START, CALIBRATION_END))
params = standardized_index_fit_params(
    calibration,
    freq=None,
    window=SPI_SCALE_MONTHS,
    dist="gamma",
    method="APP",
    zero_inflated=True,
    fitkwargs={"floc": 0},
)
spi = standardized_precipitation_index(pr=monthly_precipitation, params=params)
spi = spi.sel(time=slice(APPLICATION_START, APPLICATION_END)).rename("spi")
auxiliary = ["number_of_zeros", "number_of_notnull", "prob_of_zero"]
spi = spi.drop_vars([name for name in auxiliary if name in spi.coords])

## Save the result

In [ ]:
output = spi.to_dataset()
creation_date = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
summary = (
    f"{SPI_SCALE_MONTHS}-month Standardized Precipitation Index calculated from ERA5 {PRODUCT_TYPE} total precipitation. "
    f"Spatial domain: lat [{float(spi.lat.min()):.2f}, {float(spi.lat.max()):.2f} deg], "
    f"lon [{float(spi.lon.min()):.2f}, {float(spi.lon.max()):.2f} deg]. "
    f"Temporal coverage: {str(spi.time.values[0])[:10]} to {str(spi.time.values[-1])[:10]}. "
    f"Calibration period: {CALIBRATION_START} to {CALIBRATION_END}."
)
output["spi"].attrs.update(
    long_name=f"Standardized Precipitation Index ({SPI_SCALE_MONTHS}-month)",
    units="1",
    fit_failure_interpretation="With complete precipitation input, NaN SPI values indicate that the selected probability distribution could not be fitted for that calendar month and grid cell. This can occur in very arid regions when the calibration sample contains too few positive precipitation values, producing non-finite or non-positive distribution parameters.",
    numerical_bounds="[-8.21, 8.21]",
    numerical_bounds_interpretation="When the fitted cumulative probability is numerically equal to 0 or 1, its transformation to the standard normal distribution would produce negative or positive infinity. xclim clips these values to -8.21 or 8.21. Values at these bounds represent events beyond the numerical resolution of the fitted distribution and do not necessarily indicate a failed calibration fit.",
)
output.attrs.update(
    title="Standardized Precipitation Index for ERA5",
    summary=summary,
    creator="Marcio Cataldi <mcataldi@id.uff.br>",
    institution="Climate System Monitoring and Modeling Laboratory (LAMMOC), Universidade Federal Fluminense (UFF), Niteroi, Brazil",
    project="RiskClima",
    license="CC-BY-4.0",
    references="https://riskclima.com.br/",
    code_repository="https://github.com/lammoc-uff/cnpq-riskclima",
    Conventions="CF-1.10",
    processing_level="Processed data",
    source=f"ERA5 {PRODUCT_TYPE} total precipitation",
    keywords="spi, standardized precipitation index, ERA5, reanalysis, RiskClima",
    input_variables="tp",
    input_frequency="monthly",
    calibration_period=f"{CALIBRATION_START} to {CALIBRATION_END}",
    application_period=f"{APPLICATION_START} to {APPLICATION_END}",
    calibration_method="gamma distribution fitted independently for each calendar month and grid cell using xclim APP, floc=0, and zero-inflated precipitation.",
    compute_backend="xarray and xclim",
    spi_scale_months=SPI_SCALE_MONTHS,
    spi_distribution="gamma",
    spi_fitting_method="APP",
    spi_floc=0,
    precipitation_conversion="ERA5 monthly averaged total precipitation represents an accumulation with an effective processing period of one day. Values in metres per day were multiplied by 1000 and by the number of days in each calendar month, producing monthly accumulated precipitation in mm month-1.",
    monthly_precipitation_units="mm month-1",
    creation_date=creation_date,
    history=f"{creation_date} Computed {SPI_SCALE_MONTHS}-month SPI using xarray and xclim.",
)
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
output.to_netcdf(OUTPUT_FILE, engine="netcdf4", format="NETCDF4")
logger.info("SPI output written to %s", OUTPUT_FILE)
OUTPUT_FILE